## Making my own GPT 

In [102]:
# Asking for Prompt From User

user_prompt = input("HOW CAN I HELP YOU 🙏😘 Baby !")

print("User Prompt :- ",user_prompt)

User Prompt :-  yo how you big bro now suggest some gangsta movie like godfathers


In [103]:
# Tokenize the prompt
from transformers import AutoModel,AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

gpt_2 = AutoModel.from_pretrained("gpt2")

tokens = tokenizer(user_prompt,return_tensors="pt") #tokens form user prompt as per gpt-2

Vector_embeddings = gpt_2.get_input_embeddings()

embeddings = Vector_embeddings(tokens["input_ids"]) # embeddings from the gpt-2 model


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 11815.39it/s]


In [104]:
# Tokens and the vector representation of it
print(f"Glimpse of tokens of the user prompt :- {tokens["input_ids"][0][0:10]}")
print(f"length tokens of the user prompt :- {len(tokens["input_ids"][0])}")

print("  ")
print("-------------------------------------")
print("  ")

print(f" shape of the Embeddings of the user prompt :- {embeddings.shape}")
print(f" glimpse of the embeddigs :- {embeddings[0][0][0:5]}")

Glimpse of tokens of the user prompt :- tensor([ 8226,   703,   345,  1263,  1379,   783,  1950,   617,  7706, 38031])
length tokens of the user prompt :- 15
  
-------------------------------------
  
 shape of the Embeddings of the user prompt :- torch.Size([1, 15, 768])
 glimpse of the embeddigs :- tensor([-0.1423, -0.1747,  0.0495, -0.3202, -0.1142], grad_fn=<SliceBackward0>)


In [105]:
# add positional encoding to the Vector Embeddings
import torch
import numpy as np
def positional_encoding(length_of_ids,embeddings_lengths):
    # this takes the same shape as vector embeddings gives u positional embeddings for it

    pos = length_of_ids
    d = embeddings_lengths
    pe = torch.zeros((1,pos,d))
    for i in range(pos):
        for j in range(d//2):
            pe[0,i,2*j] = np.sin(i / 10000 ** (2 * j / d ))
            pe[0,i,2*j+1] = np.cos(i / 10000 ** (2 * j/ d ))
    return pe


In [106]:
#positional encodings
pe = positional_encoding(embeddings.shape[1],embeddings.shape[2])

In [107]:
embeddings.shape

torch.Size([1, 15, 768])

In [108]:
pe.shape

torch.Size([1, 15, 768])

In [109]:
# final input embeddings = vector_embedding + positional encoding
input_embeddings = embeddings + pe

In [110]:

print(f" shape of the input embeddings of the user prompt :- {input_embeddings.shape}")
print(f" glimpse of the input embeddigs :- {embeddings[0][0][0:5]}")

 shape of the input embeddings of the user prompt :- torch.Size([1, 15, 768])
 glimpse of the input embeddigs :- tensor([-0.1423, -0.1747,  0.0495, -0.3202, -0.1142], grad_fn=<SliceBackward0>)


## Pre Attention Done

In [116]:
input_embeddings = input_embeddings.squeeze(0)

residual_1 = input_embeddings.clone()

In [118]:
#init the wieghts
w_k = torch.rand(768,768,requires_grad=True)
w_v = torch.rand(768,768,requires_grad=True)
w_q = torch.rand(768,768,requires_grad=True)

# calculating the Query , Key ,Value
Q = input_embeddings@w_q
K = input_embeddings@w_k
V = input_embeddings@w_v

# Attention Score
Attention_Score = torch.nn.functional.softmax((Q@K.T)/np.sqrt(768),dim=-1)@V

In [119]:
# making the output of attention Block

Attention_out = Attention_Score + residual_1
Attention_out.shape


torch.Size([15, 768])

In [120]:
print(f"The shape of attention out {Attention_out.shape}")
print(f"The shape of attention out {Attention_out[0][:5]}")

The shape of attention out torch.Size([15, 768])
The shape of attention out tensor([208.9690, 208.1822, 203.0357, 213.1304, 196.5217],
       grad_fn=<SliceBackward0>)


## Attention Block Done

In [127]:
# Normalizing Before The FFN (feed forward network ) layer


Attention_out_rms = torch.nn.functional.rms_norm(Attention_out,Attention_out.shape)